# 카테고리별 백분위 기반 등급 컷오프 자동화

**목적**: 고정 컷오프(85/70/55) 대신 카테고리별 점수 분포의 백분위를 기준으로 등급을 산출  
**문제**: 카테고리마다 점수 분포가 다르므로 고정 기준 적용 시 특정 카테고리에 등급 쏠림 발생  
**해결**: 카테고리별 base_score 분포에서 백분위 컷오프를 자동 산출 → BQ 뷰로 저장 → Tableau 참조

In [1]:
import sys
sys.path.insert(0, '../../05_src/02_bigquery')

import pandas as pd
import numpy as np
from bq_client import query_to_df

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.1f}'.format)

## 1. 현황 진단: 고정 컷오프의 문제

In [2]:
# 카테고리별 base_score 분포 조회
sql_dist = """
SELECT
    category_2,
    base_score,
    final_soft_landing
FROM `daiso.v_score_distribution`
"""
df_scores = query_to_df(sql_dist)
print(f"전체 제품 수: {len(df_scores)}")
print(f"카테고리 수: {df_scores['category_2'].nunique()}")
df_scores.head()

전체 제품 수: 706
카테고리 수: 14


/opt/miniconda3/envs/py_study/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_2,base_score,final_soft_landing
0,팩/마스크,60,False
1,팩/마스크,60,False
2,팩/마스크,60,False
3,팩/마스크,60,True
4,팩/마스크,60,True


In [3]:
# 카테고리별 base_score 기술통계
cat_stats = df_scores.groupby('category_2')['base_score'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(1).sort_values('mean', ascending=False)

cat_stats

,count,mean,std,min,median,max
category_2,,,,,,
남성향수,2,52.0,4.2,49,52.0,55
클렌징/필링,70,45.2,5.7,35,44.0,55
팩/마스크,32,44.8,8.4,35,44.0,60
자외선차단제,29,42.4,3.7,36,42.0,48
기초스킨케어,189,40.9,4.6,27,42.0,48
립케어,18,40.1,5.3,35,38.0,55
남성스킨케어,10,35.3,7.4,23,37.0,45
아이메이크업,88,33.7,6.1,23,34.0,43
베이스메이크업,89,33.6,6.2,23,32.0,48


In [4]:
# 고정 컷오프(85/70/55) 적용 시 등급 분포 → 쏠림 확인
# base_score는 축2(25점), 축5(15점) 제외 → 최대 60점
# 사용자 입력까지 합산하면 최대 100점이므로, base_score 기준 고정 컷오프는 의미 없음
# → base_score 분포 내에서 상대적 위치(백분위)로 등급 부여가 합리적

def apply_fixed_grade(score):
    """기존 고정 컷오프 (base_score 60점 기준 비례 축소)"""
    if score >= 51:  # 85/100 * 60 = 51
        return 'A'
    elif score >= 42:  # 70/100 * 60 = 42
        return 'B'
    elif score >= 33:  # 55/100 * 60 = 33
        return 'C'
    return 'D'

df_scores['fixed_grade'] = df_scores['base_score'].apply(apply_fixed_grade)

# 카테고리별 고정 등급 분포
fixed_dist = df_scores.groupby(['category_2', 'fixed_grade']).size().unstack(fill_value=0)
fixed_pct = fixed_dist.div(fixed_dist.sum(axis=1), axis=0).round(3) * 100
print("=== 고정 컷오프 적용 시 카테고리별 등급 비율(%) ===")
fixed_pct

=== 고정 컷오프 적용 시 카테고리별 등급 비율(%) ===


fixed_grade,A,B,C,D
category_2,,,,
기초스킨케어,0.0,52.9,42.9,4.2
남성메이크업,0.0,0.0,0.0,100.0
남성스킨케어,0.0,30.0,30.0,40.0
남성용면도기,0.0,0.0,0.0,100.0
남성향수,50.0,50.0,0.0,0.0
립메이크업,0.0,0.0,0.0,100.0
립케어,5.6,33.3,61.1,0.0
베이스메이크업,0.0,19.1,24.7,56.2
아이메이크업,0.0,14.8,51.1,34.1


## 2. 백분위 기반 컷오프 산출

**등급 기준 (백분위)**:
- A등급: 상위 10% (>= P90)
- B등급: 상위 30% (>= P70)
- C등급: 상위 60% (>= P40)
- D등급: 하위 40% (< P40)

In [5]:
# 백분위 기준 정의
GRADE_PERCENTILES = {
    'A': 90,  # 상위 10%
    'B': 70,  # 상위 30%
    'C': 40,  # 상위 60%
    # D: 나머지
}

def compute_category_cutoffs(df, percentiles=GRADE_PERCENTILES):
    """
    카테고리별 백분위 기반 등급 컷오프 산출
    
    Parameters
    ----------
    df : DataFrame
        category_2, base_score 컬럼 포함
    percentiles : dict
        등급별 백분위 기준 (A: 90 → 상위 10%)
    
    Returns
    -------
    DataFrame
        카테고리별 컷오프 테이블
    """
    results = []
    
    for cat, group in df.groupby('category_2'):
        scores = group['base_score'].values
        n = len(scores)
        
        cutoff_a = np.percentile(scores, percentiles['A'])
        cutoff_b = np.percentile(scores, percentiles['B'])
        cutoff_c = np.percentile(scores, percentiles['C'])
        
        # 동점 처리: 컷오프가 같으면 상위 등급 우선
        # (소규모 카테고리에서 발생 가능)
        if cutoff_b == cutoff_a:
            cutoff_b = cutoff_a - 1
        if cutoff_c == cutoff_b:
            cutoff_c = cutoff_b - 1
        
        # 등급별 실제 비율 검증
        pct_a = (scores >= cutoff_a).mean() * 100
        pct_b = ((scores >= cutoff_b) & (scores < cutoff_a)).mean() * 100
        pct_c = ((scores >= cutoff_c) & (scores < cutoff_b)).mean() * 100
        pct_d = (scores < cutoff_c).mean() * 100
        
        results.append({
            'category_2': cat,
            'n_products': n,
            'score_mean': round(np.mean(scores), 1),
            'score_std': round(np.std(scores), 1),
            'cutoff_a': cutoff_a,
            'cutoff_b': cutoff_b,
            'cutoff_c': cutoff_c,
            'pct_a': round(pct_a, 1),
            'pct_b': round(pct_b, 1),
            'pct_c': round(pct_c, 1),
            'pct_d': round(pct_d, 1),
        })
    
    return pd.DataFrame(results)

df_cutoffs = compute_category_cutoffs(df_scores)
df_cutoffs.sort_values('score_mean', ascending=False)

,category_2,n_products,score_mean,score_std,cutoff_a,cutoff_b,cutoff_c,pct_a,pct_b,pct_c,pct_d
4,남성향수,2,52.0,3.0,54.4,53.2,51.4,50.0,0.0,0.0,50.0
12,클렌징/필링,70,45.2,5.6,55.0,48.0,44.0,14.3,17.1,35.7,32.9
13,팩/마스크,32,44.8,8.3,60.0,49.0,40.4,15.6,18.8,25.0,40.6
9,자외선차단제,29,42.4,3.7,48.0,45.0,42.0,17.2,24.1,20.7,37.9
0,기초스킨케어,189,40.9,4.6,48.0,43.0,40.0,13.8,29.1,18.0,39.2
6,립케어,18,40.1,5.2,44.0,43.9,38.0,33.3,0.0,33.3,33.3
2,남성스킨케어,10,35.3,7.0,43.2,38.8,35.0,10.0,20.0,30.0,40.0
8,아이메이크업,88,33.7,6.1,43.0,37.0,34.0,14.8,22.7,28.4,34.1
7,베이스메이크업,89,33.6,6.2,43.0,37.0,31.0,16.9,16.9,31.5,34.8
10,치크/하이라이터,61,25.2,5.0,33.0,27.0,24.0,14.8,27.9,39.3,18.0


In [6]:
# 백분위 등급 적용 후 SL 제품의 등급 분포 검증
# → A등급에 SL 제품이 집중되어야 모델이 유의미

def apply_percentile_grade(row, cutoffs_df):
    """카테고리별 백분위 컷오프로 등급 부여"""
    cat = row['category_2']
    score = row['base_score']
    cat_cutoffs = cutoffs_df[cutoffs_df['category_2'] == cat].iloc[0]
    
    if score >= cat_cutoffs['cutoff_a']:
        return 'A'
    elif score >= cat_cutoffs['cutoff_b']:
        return 'B'
    elif score >= cat_cutoffs['cutoff_c']:
        return 'C'
    return 'D'

df_scores['pct_grade'] = df_scores.apply(
    lambda row: apply_percentile_grade(row, df_cutoffs), axis=1
)

# SL 제품의 등급별 분포
sl_grade_dist = df_scores.groupby(['pct_grade', 'final_soft_landing']).size().unstack(fill_value=0)
sl_grade_dist['sl_rate'] = (sl_grade_dist[True] / sl_grade_dist.sum(axis=1) * 100).round(1)
print("=== 백분위 등급별 SL 제품 비율 ===")
sl_grade_dist

=== 백분위 등급별 SL 제품 비율 ===


final_soft_landing,False,True,sl_rate
pct_grade,,,
A,64,42,39.6
B,169,44,20.7
C,129,30,18.9
D,186,42,18.4


In [7]:
# 고정 vs 백분위: 카테고리별 등급 분포 비교
comparison = df_scores.groupby('category_2').apply(
    lambda g: pd.Series({
        'n': len(g),
        'fixed_A%': (g['fixed_grade'] == 'A').mean() * 100,
        'fixed_D%': (g['fixed_grade'] == 'D').mean() * 100,
        'pct_A%': (g['pct_grade'] == 'A').mean() * 100,
        'pct_D%': (g['pct_grade'] == 'D').mean() * 100,
    })
).round(1)

print("=== 고정 vs 백분위 등급 비교 ===")
print("(고정 컷오프에서 A가 0%이거나 D가 80%+인 카테고리 = 쏠림 문제)")
comparison.sort_values('fixed_D%', ascending=False)

=== 고정 vs 백분위 등급 비교 ===
(고정 컷오프에서 A가 0%이거나 D가 80%+인 카테고리 = 쏠림 문제)


/var/folders/3b/mp7mjwzj0y38km8nrsylgq180000gn/T/ipykernel_92956/4036717078.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  comparison = df_scores.groupby('category_2').apply(


,n,fixed_A%,fixed_D%,pct_A%,pct_D%
category_2,,,,,
남성메이크업,1.0,0.0,100.0,100.0,0.0
남성용면도기,1.0,0.0,100.0,100.0,0.0
립메이크업,115.0,0.0,100.0,10.4,20.9
클렌징/쉐이빙,1.0,0.0,100.0,100.0,0.0
치크/하이라이터,61.0,0.0,85.2,14.8,18.0
베이스메이크업,89.0,0.0,56.2,16.9,34.8
남성스킨케어,10.0,0.0,40.0,10.0,40.0
아이메이크업,88.0,0.0,34.1,14.8,34.1
기초스킨케어,189.0,0.0,4.2,13.8,39.2


## 3. 민감도 분석: 백분위 기준 조정

In [8]:
# 백분위 기준 변경 시 SL 포착률 변화
configs = [
    {'name': '엄격 (5/20/50)',  'A': 95, 'B': 80, 'C': 50},
    {'name': '기본 (10/30/60)', 'A': 90, 'B': 70, 'C': 40},
    {'name': '관대 (15/40/70)', 'A': 85, 'B': 60, 'C': 30},
]

sensitivity = []
for cfg in configs:
    pcts = {'A': cfg['A'], 'B': cfg['B'], 'C': cfg['C']}
    tmp_cutoffs = compute_category_cutoffs(df_scores, pcts)
    
    df_scores['tmp_grade'] = df_scores.apply(
        lambda row: apply_percentile_grade(
            row, tmp_cutoffs
        ), axis=1
    )
    
    sl_in_a = df_scores[
        (df_scores['tmp_grade'] == 'A') & (df_scores['final_soft_landing'] == True)
    ].shape[0]
    total_sl = df_scores['final_soft_landing'].sum()
    total_a = (df_scores['tmp_grade'] == 'A').sum()
    
    sensitivity.append({
        '기준': cfg['name'],
        'A등급_수': total_a,
        'A등급_SL수': sl_in_a,
        'A등급_SL비율%': round(sl_in_a / total_a * 100, 1) if total_a > 0 else 0,
        'SL_포착률%': round(sl_in_a / total_sl * 100, 1) if total_sl > 0 else 0,
    })

df_sens = pd.DataFrame(sensitivity)
print("=== 민감도 분석: 백분위 기준별 A등급 SL 포착률 ===")
df_sens

=== 민감도 분석: 백분위 기준별 A등급 SL 포착률 ===


,기준,A등급_수,A등급_SL수,A등급_SL비율%,SL_포착률%
0,엄격 (5/20/50),96,38,39.6,24.1
1,기본 (10/30/60),106,42,39.6,26.6
2,관대 (15/40/70),159,53,33.3,33.5


## 4. 최종 컷오프 테이블 확정 및 출력

In [9]:
# 최종 컷오프 테이블 (BQ 뷰 및 Tableau 참조용)
final_cutoffs = df_cutoffs[[
    'category_2', 'n_products', 'score_mean',
    'cutoff_a', 'cutoff_b', 'cutoff_c',
    'pct_a', 'pct_b', 'pct_c', 'pct_d'
]].sort_values('category_2')

print("=== 최종 카테고리별 등급 컷오프 ===")
print(f"기준: A >= P{GRADE_PERCENTILES['A']} / B >= P{GRADE_PERCENTILES['B']} / C >= P{GRADE_PERCENTILES['C']} / D: 나머지")
print()
final_cutoffs

=== 최종 카테고리별 등급 컷오프 ===
기준: A >= P90 / B >= P70 / C >= P40 / D: 나머지



,category_2,n_products,score_mean,cutoff_a,cutoff_b,cutoff_c,pct_a,pct_b,pct_c,pct_d
0,기초스킨케어,189,40.9,48.0,43.0,40.0,13.8,29.1,18.0,39.2
1,남성메이크업,1,17.0,17.0,16.0,17.0,100.0,0.0,0.0,0.0
2,남성스킨케어,10,35.3,43.2,38.8,35.0,10.0,20.0,30.0,40.0
3,남성용면도기,1,12.0,12.0,11.0,12.0,100.0,0.0,0.0,0.0
4,남성향수,2,52.0,54.4,53.2,51.4,50.0,0.0,0.0,50.0
5,립메이크업,115,17.4,22.8,18.0,17.0,10.4,68.7,0.0,20.9
6,립케어,18,40.1,44.0,43.9,38.0,33.3,0.0,33.3,33.3
7,베이스메이크업,89,33.6,43.0,37.0,31.0,16.9,16.9,31.5,34.8
8,아이메이크업,88,33.7,43.0,37.0,34.0,14.8,22.7,28.4,34.1
9,자외선차단제,29,42.4,48.0,45.0,42.0,17.2,24.1,20.7,37.9


## 5. BQ 뷰 SQL 생성

아래 SQL을 `tableau_scorecard_views.sql`에 추가하여 Tableau에서 라이브 참조

In [11]:
# BQ 뷰 SQL 자동 생성
bq_sql = f"""
-- =============================================
-- VIEW 9: 카테고리별 백분위 등급 컷오프
-- =============================================
-- 용도: Tableau 등급 산출 시 카테고리별 동적 컷오프 참조
-- 로직: base_score(축2,5 제외)의 카테고리별 백분위 산출
--       A >= P{GRADE_PERCENTILES['A']} / B >= P{GRADE_PERCENTILES['B']} / C >= P{GRADE_PERCENTILES['C']} / D: 나머지
-- =============================================
CREATE OR REPLACE VIEW `daiso.v_category_grade_cutoffs` AS
WITH score_base AS (
  SELECT
    category_2,
    base_score
  FROM `daiso.v_score_distribution`
),

category_percentiles AS (
  SELECT
    category_2,
    COUNT(*) AS n_products,
    ROUND(AVG(base_score), 1) AS score_mean,
    ROUND(STDDEV(base_score), 1) AS score_std,
    -- 백분위 컷오프
    APPROX_QUANTILES(base_score, 100)[OFFSET({GRADE_PERCENTILES['A']})] AS cutoff_a,
    APPROX_QUANTILES(base_score, 100)[OFFSET({GRADE_PERCENTILES['B']})] AS cutoff_b,
    APPROX_QUANTILES(base_score, 100)[OFFSET({GRADE_PERCENTILES['C']})] AS cutoff_c
  FROM score_base
  GROUP BY category_2
)

SELECT
  category_2,
  n_products,
  score_mean,
  score_std,
  cutoff_a,
  -- 동점 보정: B 컷오프가 A와 같으면 A-1
  CASE WHEN cutoff_b >= cutoff_a THEN cutoff_a - 1 ELSE cutoff_b END AS cutoff_b,
  -- 동점 보정: C 컷오프가 B와 같으면 B-1
  CASE
    WHEN cutoff_c >= CASE WHEN cutoff_b >= cutoff_a THEN cutoff_a - 1 ELSE cutoff_b END
    THEN CASE WHEN cutoff_b >= cutoff_a THEN cutoff_a - 1 ELSE cutoff_b END - 1
    ELSE cutoff_c
  END AS cutoff_c
FROM category_percentiles
ORDER BY category_2;
"""

print(bq_sql)


-- =============================================
-- VIEW 9: 카테고리별 백분위 등급 컷오프
-- =============================================
-- 용도: Tableau 등급 산출 시 카테고리별 동적 컷오프 참조
-- 로직: base_score(축2,5 제외)의 카테고리별 백분위 산출
--       A >= P90 / B >= P70 / C >= P40 / D: 나머지
-- =============================================
CREATE OR REPLACE VIEW `daiso.v_category_grade_cutoffs` AS
WITH score_base AS (
  SELECT
    category_2,
    base_score
  FROM `daiso.v_score_distribution`
),

category_percentiles AS (
  SELECT
    category_2,
    COUNT(*) AS n_products,
    ROUND(AVG(base_score), 1) AS score_mean,
    ROUND(STDDEV(base_score), 1) AS score_std,
    -- 백분위 컷오프
    APPROX_QUANTILES(base_score, 100)[OFFSET(90)] AS cutoff_a,
    APPROX_QUANTILES(base_score, 100)[OFFSET(70)] AS cutoff_b,
    APPROX_QUANTILES(base_score, 100)[OFFSET(40)] AS cutoff_c
  FROM score_base
  GROUP BY category_2
)

SELECT
  category_2,
  n_products,
  score_mean,
  score_std,
  cutoff_a,
  -- 동점 보정: B 컷오프가 A와 같으면 A-1
  CASE WHEN cu

In [12]:
# BQ 뷰 실행 (선택)
query_to_df(bq_sql)  # CREATE VIEW는 to_dataframe() 불가 → 아래 방식 사용

from bq_client import get_client
client = get_client()
client.query(bq_sql).result()
print("v_category_grade_cutoffs 뷰 생성 완료")

v_category_grade_cutoffs 뷰 생성 완료


## 6. Tableau 계산된 필드 수식

Tableau에서 `v_category_grade_cutoffs`를 데이터 소스로 추가한 뒤,  
`p_category_2` 파라미터와 조인하여 동적 컷오프를 참조한다.

In [13]:
tableau_formulas = """
========================================
Tableau 계산된 필드 (기존 수식 교체)
========================================

1) 데이터 소스 추가:
   - v_category_grade_cutoffs를 BQ 라이브 연결로 추가
   - category_2 기준으로 v_category_scorecard_ref와 관계 설정

2) [등급] 수식 교체 (기존 고정 → 백분위 동적):

// [등급_v3] — 카테고리별 백분위 기반
IF [총점] >= [cutoff_a] THEN 'A'
ELSEIF [총점] >= [cutoff_b] THEN 'B'
ELSEIF [총점] >= [cutoff_c] THEN 'C'
ELSE 'D'
END

※ [cutoff_a], [cutoff_b], [cutoff_c]는
   v_category_grade_cutoffs에서 p_category_2 파라미터에
   매칭되는 행의 값을 LOD로 가져온다:

// [cutoff_a]
{FIXED : MIN(
  IF [category_2] = [p_category_2] THEN [cutoff_a_raw] END
)}

// [cutoff_b]
{FIXED : MIN(
  IF [category_2] = [p_category_2] THEN [cutoff_b_raw] END
)}

// [cutoff_c]
{FIXED : MIN(
  IF [category_2] = [p_category_2] THEN [cutoff_c_raw] END
)}

3) [등급_설명_v3] 수식:

// [등급_설명_v3]
IF [등급_v3] = 'A' THEN
  '입점 적극 추천 — 해당 카테고리 상위 ' + STR(ROUND(100 - 90)) + '% (연착륙 가능성 매우 높음)'
ELSEIF [등급_v3] = 'B' THEN
  '입점 추천 — 해당 카테고리 상위 ' + STR(ROUND(100 - 70)) + '% (조건부 성공 가능)'
ELSEIF [등급_v3] = 'C' THEN
  '입점 검토 필요 — 해당 카테고리 상위 ' + STR(ROUND(100 - 40)) + '% (리스크 요인 존재)'
ELSE
  '입점 재고 — 해당 카테고리 하위 ' + STR(ROUND(40)) + '%'
END

4) [백분위_v3] — 카테고리 내 상대 위치:

// v_score_distribution에서 같은 카테고리 내 base_score 비교
COUNTD(IF [sd_category_2] = [p_category_2]
       AND [sd_base_score] <= [총점]
       THEN [sd_product_code] END)
/
COUNTD(IF [sd_category_2] = [p_category_2]
       THEN [sd_product_code] END)
* 100

※ sd_ 접두사는 v_score_distribution 데이터 소스의 필드
"""

print(tableau_formulas)


Tableau 계산된 필드 (기존 수식 교체)

1) 데이터 소스 추가:
   - v_category_grade_cutoffs를 BQ 라이브 연결로 추가
   - category_2 기준으로 v_category_scorecard_ref와 관계 설정

2) [등급] 수식 교체 (기존 고정 → 백분위 동적):

// [등급_v3] — 카테고리별 백분위 기반
IF [총점] >= [cutoff_a] THEN 'A'
ELSEIF [총점] >= [cutoff_b] THEN 'B'
ELSEIF [총점] >= [cutoff_c] THEN 'C'
ELSE 'D'
END

※ [cutoff_a], [cutoff_b], [cutoff_c]는
   v_category_grade_cutoffs에서 p_category_2 파라미터에
   매칭되는 행의 값을 LOD로 가져온다:

// [cutoff_a]
{FIXED : MIN(
  IF [category_2] = [p_category_2] THEN [cutoff_a_raw] END
)}

// [cutoff_b]
{FIXED : MIN(
  IF [category_2] = [p_category_2] THEN [cutoff_b_raw] END
)}

// [cutoff_c]
{FIXED : MIN(
  IF [category_2] = [p_category_2] THEN [cutoff_c_raw] END
)}

3) [등급_설명_v3] 수식:

// [등급_설명_v3]
IF [등급_v3] = 'A' THEN
  '입점 적극 추천 — 해당 카테고리 상위 ' + STR(ROUND(100 - 90)) + '% (연착륙 가능성 매우 높음)'
ELSEIF [등급_v3] = 'B' THEN
  '입점 추천 — 해당 카테고리 상위 ' + STR(ROUND(100 - 70)) + '% (조건부 성공 가능)'
ELSEIF [등급_v3] = 'C' THEN
  '입점 검토 필요 — 해당 카테고리 상위 ' + STR(ROUND(100 - 40)) + '% 

## 7. 검증: 등급 변경 영향도

In [14]:
# 고정 → 백분위 전환 시 등급이 변하는 제품 수
df_scores['grade_changed'] = df_scores['fixed_grade'] != df_scores['pct_grade']

change_summary = pd.DataFrame({
    '총 제품': [len(df_scores)],
    '등급 변경': [df_scores['grade_changed'].sum()],
    '변경 비율%': [round(df_scores['grade_changed'].mean() * 100, 1)],
})
print("=== 등급 변경 영향도 ===")
print(change_summary.to_string(index=False))
print()

# 등급 이동 매트릭스 (고정 → 백분위)
migration = pd.crosstab(
    df_scores['fixed_grade'],
    df_scores['pct_grade'],
    margins=True
)
migration.index.name = '고정(기존)'
migration.columns.name = '백분위(신규)'
print("=== 등급 이동 매트릭스 ===")
migration

=== 등급 변경 영향도 ===
 총 제품  등급 변경  변경 비율%
  706    445    63.0

=== 등급 이동 매트릭스 ===


백분위(신규),A,B,C,D,All
고정(기존),,,,,
A,17,7,0,0,24
B,65,77,57,5,204
C,9,33,59,115,216
D,15,96,43,108,262
All,106,213,159,228,706
